## Train a Multimodal Hybrid CNN + Landmark Transformer on 100-Driver Dataset

Single MobileNetV3-Small backbone + geometric landmark features + lightweight transformer fusion.

## 0) Install packages

In [ ]:
# %pip install -U pip setuptools wheel
# %pip install torch torchvision --index-url https://download.pytorch.org/whl/cu130
# %pip install numpy pandas scikit-learn pillow tqdm matplotlib

## 1) Imports

In [ ]:
import os
import json
import math
import random
import time
from collections import Counter, defaultdict
from dataclasses import dataclass
from pathlib import Path
from typing import Any, Dict, List, Optional

import numpy as np
import pandas as pd
from PIL import Image, ImageFile

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

from sklearn.metrics import accuracy_score, f1_score, classification_report, confusion_matrix

import matplotlib.pyplot as plt
from tqdm.auto import tqdm

from torchvision import transforms
from torchvision.transforms import InterpolationMode
from torchvision.models import mobilenet_v3_small, MobileNet_V3_Small_Weights

ImageFile.LOAD_TRUNCATED_IMAGES = True

## 2) Config

In [ ]:
DATA_ROOT = Path(r"../../Datasets/100-driver")
SETTING = "Traditional-setting"
TIME_OF_DAY = "Day"
CAMERA = "Cam1"
SPLIT_PREFIX = None
IMG_ROOT_OVERRIDE = None
LANDMARK_PATH_OVERRIDE = None

@dataclass
class TrainConfig:
    # Training
    epochs: int = 100
    backbone_lr: float = 5e-5
    head_lr: float = 5e-4
    weight_decay: float = 1e-2
    batch_size: int = 32
    num_workers: int = 10
    img_size: int = 224
    warm_epochs: int = 5
    save_every: int = 20
    seed: int = 42

    # Transformer
    embed_dim: int = 128
    num_heads: int = 4
    num_layers: int = 4
    ff_dim: int = 384
    dropout: float = 0.15

    # Landmark config
    pose_points: int = 33
    hand_points: int = 21
    face_points: int = 478
    landmark_dims: int = 3

    # Geometric features computed from landmarks (6 head + 2 eye + 1 mouth + 12 pose + 8 hand = 29)
    num_geo_features: int = 29

    out_dir: Path = Path(f"../best_models/100-driver/multimodal_v2/{CAMERA}_{TIME_OF_DAY}")
    best_name: str = f"hybrid_transformer_100driver_{CAMERA}_{TIME_OF_DAY}_best.pt"

cfg = TrainConfig()
cfg.out_dir.mkdir(parents=True, exist_ok=True)
print("Config:", cfg)

## 3) Utils

In [ ]:
def seed_everything(seed: int = 42):
    random.seed(seed)
    os.environ["PYTHONHASHSEED"] = str(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = False
    torch.backends.cudnn.benchmark = True

def cuda_sanity_check():
    print("torch:", torch.__version__)
    try:
        import torchvision
        print("torchvision:", torchvision.__version__)
    except Exception as e:
        print("torchvision import failed:", e)

    print("torch.version.cuda:", torch.version.cuda)
    print("CUDA available:", torch.cuda.is_available())

    if torch.cuda.is_available():
        idx = torch.cuda.current_device()
        props = torch.cuda.get_device_properties(idx)
        print("GPU:", props.name)
        print("Compute capability:", f"sm_{props.major}{props.minor}")
        print("Total VRAM (GB):", round(props.total_memory / (1024**3), 2))
        x = torch.randn(1024, 1024, device="cuda")
        y = x @ x
        print("Matmul OK. Mean:", float(y.mean()))
    else:
        print("No CUDA GPU detected, falling back to CPU.")

def parse_split_file(txt_path: Path):
    records = []
    with open(txt_path, "r", encoding="utf-8") as f:
        for line_no, line in enumerate(f, start=1):
            line = line.strip()
            if not line:
                continue
            parts = line.split()
            if len(parts) < 3:
                print(f"Skipping malformed line {line_no} in {txt_path}: {line!r}")
                continue

            sample_id = parts[0]
            rel_path = parts[1]
            label = int(parts[2])
            records.append({"sample_id": sample_id, "rel_path": rel_path, "label": label})
    return records

def infer_class_names(records):
    names_by_label = defaultdict(list)
    for rec in records:
        rel = Path(rec["rel_path"])
        folder_name = rel.parent.name if rel.parent.name else f"class_{rec['label']}"
        names_by_label[rec["label"]].append(folder_name)

    max_label = max(r["label"] for r in records)
    ordered_names = []
    for i in range(max_label + 1):
        if i in names_by_label:
            ordered_names.append(Counter(names_by_label[i]).most_common(1)[0][0])
        else:
            ordered_names.append(f"class_{i}")
    return ordered_names

def normalize_lookup_key(value: Any) -> Optional[str]:
    if value is None:
        return None
    text = str(value).strip()
    if not text:
        return None
    return text.replace("\\", "/")

def register_entry(lookup: Dict[str, Any], key: Any, entry: Any):
    norm = normalize_lookup_key(key)
    if norm is None:
        return
    lookup.setdefault(norm, entry)
    p = Path(norm)
    lookup.setdefault(p.as_posix(), entry)
    lookup.setdefault(p.name, entry)
    lookup.setdefault(p.stem, entry)

def load_landmark_lookup(landmark_path: Optional[Path]) -> Dict[str, Any]:
    if landmark_path is None or not Path(landmark_path).exists():
        return {}

    with open(landmark_path, "r", encoding="utf-8") as f:
        raw = json.load(f)

    lookup: Dict[str, Any] = {}

    if isinstance(raw, dict):
        if all(isinstance(v, dict) for v in raw.values()):
            for key, value in raw.items():
                register_entry(lookup, key, value)
                if isinstance(value, dict):
                    for extra_key in (
                        value.get("rel_path"), value.get("image_path"),
                        value.get("img_path"), value.get("path"),
                        value.get("sample_id"), value.get("id"),
                        value.get("file_name"), value.get("filename"),
                    ):
                        register_entry(lookup, extra_key, value)
        else:
            for extra_key in (
                raw.get("rel_path"), raw.get("image_path"),
                raw.get("img_path"), raw.get("path"),
                raw.get("sample_id"), raw.get("id"),
                raw.get("file_name"), raw.get("filename"),
            ):
                register_entry(lookup, extra_key, raw)
    elif isinstance(raw, list):
        for item in raw:
            if not isinstance(item, dict):
                continue
            for extra_key in (
                item.get("rel_path"), item.get("image_path"),
                item.get("img_path"), item.get("path"),
                item.get("sample_id"), item.get("id"),
                item.get("file_name"), item.get("filename"),
            ):
                register_entry(lookup, extra_key, item)

    return lookup

def resolve_landmark_path(override_path, data_root, time_of_day, camera):
    candidates = []
    if override_path:
        candidates.append(Path(override_path))

    camera_lower = camera.lower()
    time_lower = time_of_day.lower()
    candidates.extend([
        data_root / f"driver_landmarks_{time_lower}_{camera_lower}.json",
        data_root / f"driver_landmarks_{time_lower}_{camera}.json",
        data_root / f"driver_landmarks_{camera_lower}_{time_lower}.json",
        data_root / f"driver_landmarks_{time_lower}.json",
        data_root / "driver_landmarks.json",
    ])

    for cand in candidates:
        if cand.exists():
            return cand
    return None

seed_everything(cfg.seed)
cuda_sanity_check()

## 4) Parse split files and define training directories

In [ ]:
split_root = DATA_ROOT / "data-splits" / "data-splits" / SETTING / TIME_OF_DAY / CAMERA

if SPLIT_PREFIX is not None:
    prefix = SPLIT_PREFIX
elif SETTING.lower() == "traditional-setting":
    prefix = TIME_OF_DAY[0] + CAMERA[len(CAMERA) - 1]
else:
    raise ValueError("Set SPLIT_PREFIX explicitly for non-traditional settings.")

train_file = prefix + "_train.txt"
val_file = prefix + "_val.txt"
test_file = prefix + "_test.txt"
TRAIN_TXT = split_root / train_file
VAL_TXT = split_root / val_file
TEST_TXT = split_root / test_file

print("Train split:", TRAIN_TXT)
print("Val split:  ", VAL_TXT)
print("Test split: ", TEST_TXT)

train_records = parse_split_file(TRAIN_TXT)
val_records = parse_split_file(VAL_TXT)

CLASS_NAMES = infer_class_names(train_records + val_records)
NUM_CLASSES = len(CLASS_NAMES)
print(f"Classes ({NUM_CLASSES}):", CLASS_NAMES)
print(f"Train samples: {len(train_records)}, Val samples: {len(val_records)}")

if IMG_ROOT_OVERRIDE:
    IMG_ROOT = Path(IMG_ROOT_OVERRIDE)
else:
    IMG_ROOT = DATA_ROOT / TIME_OF_DAY / CAMERA

LANDMARK_PATH = resolve_landmark_path(LANDMARK_PATH_OVERRIDE, DATA_ROOT, TIME_OF_DAY, CAMERA)
print("Image root:", IMG_ROOT)
print("Landmark path:", LANDMARK_PATH)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

## 5) Transforms

In [ ]:
mean = [0.485, 0.456, 0.406]
std = [0.229, 0.224, 0.225]

train_tfms = transforms.Compose([
    transforms.Resize((cfg.img_size, cfg.img_size)),
    transforms.RandomResizedCrop(
        cfg.img_size,
        scale=(0.85, 1.0),
        ratio=(0.9, 1.1),
        interpolation=InterpolationMode.BILINEAR,
    ),
    transforms.RandomHorizontalFlip(p=0.0),  # disabled: left/right classes are distinct
    transforms.ColorJitter(brightness=0.3, contrast=0.3, saturation=0.15, hue=0.03),
    transforms.RandomRotation(10),
    transforms.ToTensor(),
    transforms.Normalize(mean=mean, std=std),
    transforms.RandomErasing(p=0.2),
])

val_tfms = transforms.Compose([
    transforms.Resize((cfg.img_size, cfg.img_size)),
    transforms.ToTensor(),
    transforms.Normalize(mean=mean, std=std),
])

print("img_size:", cfg.img_size)

## 6) Geometric Feature Extraction + Dataset

In [ ]:
# ---------------------------------------------------------------------------
# Geometric Feature Extraction from MediaPipe Landmarks
# ---------------------------------------------------------------------------
# Instead of feeding raw (x,y,z) into the transformer, we compute meaningful
# geometric features that encode body-part relationships relevant to distraction.
# This is the key differentiator from the DKT paper (raw keypoints only) and
# from pure CNN approaches (no structural information).
# ---------------------------------------------------------------------------

class GeometricFeatureExtractor:
    """
    Computes ~38 geometric features from MediaPipe face, hand, and pose landmarks.

    Features include:
    - Head pose (yaw, pitch, roll approximations)
    - Eye aspect ratios (EAR) for drowsiness
    - Mouth aspect ratio (MAR) for yawning/talking
    - Hand-face distances (phone use indicator)
    - Hand positions relative to body (reaching, texting)
    - Shoulder/body orientation
    - Hand elevation and spread
    """

    # MediaPipe face mesh key indices
    NOSE_TIP = 1
    CHIN = 152
    LEFT_EYE_OUTER = 33
    RIGHT_EYE_OUTER = 263
    LEFT_EYE_INNER = 133
    RIGHT_EYE_INNER = 362
    LEFT_MOUTH = 61
    RIGHT_MOUTH = 291
    TOP_LIP = 13
    BOTTOM_LIP = 14
    FOREHEAD = 10

    # EAR landmarks (upper/lower eyelid)
    LEFT_EYE_TOP = 159
    LEFT_EYE_BOTTOM = 145
    LEFT_EYE_LEFT = 33
    LEFT_EYE_RIGHT = 133
    RIGHT_EYE_TOP = 386
    RIGHT_EYE_BOTTOM = 374
    RIGHT_EYE_LEFT = 362
    RIGHT_EYE_RIGHT = 263

    # MAR landmarks
    MOUTH_TOP = 13
    MOUTH_BOTTOM = 14
    MOUTH_LEFT = 61
    MOUTH_RIGHT = 291

    # MediaPipe pose indices (upper body)
    POSE_NOSE = 0
    POSE_LEFT_SHOULDER = 11
    POSE_RIGHT_SHOULDER = 12
    POSE_LEFT_ELBOW = 13
    POSE_RIGHT_ELBOW = 14
    POSE_LEFT_WRIST = 15
    POSE_RIGHT_WRIST = 16
    POSE_LEFT_HIP = 23
    POSE_RIGHT_HIP = 24

    # Hand indices
    HAND_WRIST = 0
    HAND_INDEX_TIP = 8
    HAND_MIDDLE_TIP = 12
    HAND_PINKY_TIP = 20

    @staticmethod
    def _safe_get(points, idx, valid, dims=3):
        if idx < len(points) and idx < len(valid) and valid[idx]:
            return points[idx, :dims]
        return None

    @staticmethod
    def _dist(a, b):
        if a is None or b is None:
            return 0.0
        return float(np.linalg.norm(a - b))

    @staticmethod
    def _ear(top, bottom, left, right):
        vertical = GeometricFeatureExtractor._dist(top, bottom)
        horizontal = GeometricFeatureExtractor._dist(left, right)
        if horizontal < 1e-6:
            return 0.3  # default open
        return vertical / horizontal

    @staticmethod
    def _mar(top, bottom, left, right):
        vertical = GeometricFeatureExtractor._dist(top, bottom)
        horizontal = GeometricFeatureExtractor._dist(left, right)
        if horizontal < 1e-6:
            return 0.1  # default closed
        return vertical / horizontal

    @classmethod
    def compute(cls, face_pts, face_valid, pose_pts, pose_valid,
                left_hand_pts, left_valid, right_hand_pts, right_valid):
        """
        Compute geometric features. All inputs are numpy arrays.
        Returns: np.array of shape (num_geo_features,), validity mask of same shape.
        """
        g = cls._safe_get
        feats = []
        valid_mask = []

        # --- Head pose from face landmarks (6 features) ---
        nose = g(face_pts, cls.NOSE_TIP, face_valid, 3)
        chin = g(face_pts, cls.CHIN, face_valid, 3)
        left_eye_o = g(face_pts, cls.LEFT_EYE_OUTER, face_valid, 3)
        right_eye_o = g(face_pts, cls.RIGHT_EYE_OUTER, face_valid, 3)
        forehead = g(face_pts, cls.FOREHEAD, face_valid, 3)
        left_mouth = g(face_pts, cls.LEFT_MOUTH, face_valid, 3)
        right_mouth = g(face_pts, cls.RIGHT_MOUTH, face_valid, 3)

        has_face = nose is not None and chin is not None

        # Yaw approx: difference in x between nose and midpoint of eyes
        if has_face and left_eye_o is not None and right_eye_o is not None:
            eye_mid_x = (left_eye_o[0] + right_eye_o[0]) / 2
            yaw = float(nose[0] - eye_mid_x)
            feats.append(yaw)
            valid_mask.append(True)
        else:
            feats.append(0.0)
            valid_mask.append(False)

        # Pitch approx: vertical angle nose-to-chin vs nose-to-forehead
        if has_face and forehead is not None:
            up_dist = cls._dist(nose, forehead)
            down_dist = cls._dist(nose, chin)
            pitch = (down_dist - up_dist) / max(down_dist + up_dist, 1e-6)
            feats.append(pitch)
            valid_mask.append(True)
        else:
            feats.append(0.0)
            valid_mask.append(False)

        # Roll approx: angle of eye line
        if left_eye_o is not None and right_eye_o is not None:
            dx = right_eye_o[0] - left_eye_o[0]
            dy = right_eye_o[1] - left_eye_o[1]
            roll = float(np.arctan2(dy, dx + 1e-8))
            feats.append(roll)
            valid_mask.append(True)
        else:
            feats.append(0.0)
            valid_mask.append(False)

        # Nose position (normalized x, y, z) - 3 features
        if nose is not None:
            feats.extend([float(nose[0]), float(nose[1]), float(nose[2]) if len(nose) > 2 else 0.0])
            valid_mask.extend([True, True, True])
        else:
            feats.extend([0.5, 0.5, 0.0])
            valid_mask.extend([False, False, False])

        # --- Eye Aspect Ratios (2 features) ---
        left_ear = cls._ear(
            g(face_pts, cls.LEFT_EYE_TOP, face_valid, 2),
            g(face_pts, cls.LEFT_EYE_BOTTOM, face_valid, 2),
            g(face_pts, cls.LEFT_EYE_LEFT, face_valid, 2),
            g(face_pts, cls.LEFT_EYE_RIGHT, face_valid, 2),
        )
        right_ear = cls._ear(
            g(face_pts, cls.RIGHT_EYE_TOP, face_valid, 2),
            g(face_pts, cls.RIGHT_EYE_BOTTOM, face_valid, 2),
            g(face_pts, cls.RIGHT_EYE_LEFT, face_valid, 2),
            g(face_pts, cls.RIGHT_EYE_RIGHT, face_valid, 2),
        )
        feats.extend([left_ear, right_ear])
        valid_mask.extend([
            g(face_pts, cls.LEFT_EYE_TOP, face_valid) is not None,
            g(face_pts, cls.RIGHT_EYE_TOP, face_valid) is not None,
        ])

        # --- Mouth Aspect Ratio (1 feature) ---
        mar = cls._mar(
            g(face_pts, cls.MOUTH_TOP, face_valid, 2),
            g(face_pts, cls.MOUTH_BOTTOM, face_valid, 2),
            g(face_pts, cls.MOUTH_LEFT, face_valid, 2),
            g(face_pts, cls.MOUTH_RIGHT, face_valid, 2),
        )
        feats.append(mar)
        valid_mask.append(g(face_pts, cls.MOUTH_TOP, face_valid) is not None)

        # --- Pose features (12 features) ---
        p_nose = g(pose_pts, cls.POSE_NOSE, pose_valid, 3)
        l_shoulder = g(pose_pts, cls.POSE_LEFT_SHOULDER, pose_valid, 3)
        r_shoulder = g(pose_pts, cls.POSE_RIGHT_SHOULDER, pose_valid, 3)
        l_elbow = g(pose_pts, cls.POSE_LEFT_ELBOW, pose_valid, 3)
        r_elbow = g(pose_pts, cls.POSE_RIGHT_ELBOW, pose_valid, 3)
        l_wrist = g(pose_pts, cls.POSE_LEFT_WRIST, pose_valid, 3)
        r_wrist = g(pose_pts, cls.POSE_RIGHT_WRIST, pose_valid, 3)
        l_hip = g(pose_pts, cls.POSE_LEFT_HIP, pose_valid, 3)
        r_hip = g(pose_pts, cls.POSE_RIGHT_HIP, pose_valid, 3)

        # Shoulder width (normalization reference)
        shoulder_w = cls._dist(l_shoulder, r_shoulder)
        if shoulder_w < 1e-6:
            shoulder_w = 1.0

        # Body lean: shoulder midpoint x vs hip midpoint x
        if l_shoulder is not None and r_shoulder is not None and l_hip is not None and r_hip is not None:
            shoulder_mid_x = (l_shoulder[0] + r_shoulder[0]) / 2
            hip_mid_x = (l_hip[0] + r_hip[0]) / 2
            body_lean = (shoulder_mid_x - hip_mid_x) / shoulder_w
            feats.append(body_lean)
            valid_mask.append(True)
        else:
            feats.append(0.0)
            valid_mask.append(False)

        # Shoulder roll
        if l_shoulder is not None and r_shoulder is not None:
            s_dx = r_shoulder[0] - l_shoulder[0]
            s_dy = r_shoulder[1] - l_shoulder[1]
            s_roll = float(np.arctan2(s_dy, s_dx + 1e-8))
            feats.append(s_roll)
            valid_mask.append(True)
        else:
            feats.append(0.0)
            valid_mask.append(False)

        # Left wrist elevation relative to shoulder
        if l_wrist is not None and l_shoulder is not None:
            feats.append((l_shoulder[1] - l_wrist[1]) / shoulder_w)
            valid_mask.append(True)
        else:
            feats.append(0.0)
            valid_mask.append(False)

        # Right wrist elevation relative to shoulder
        if r_wrist is not None and r_shoulder is not None:
            feats.append((r_shoulder[1] - r_wrist[1]) / shoulder_w)
            valid_mask.append(True)
        else:
            feats.append(0.0)
            valid_mask.append(False)

        # Left elbow angle
        if l_shoulder is not None and l_elbow is not None and l_wrist is not None:
            v1 = l_shoulder[:2] - l_elbow[:2]
            v2 = l_wrist[:2] - l_elbow[:2]
            cos_a = np.dot(v1, v2) / (np.linalg.norm(v1) * np.linalg.norm(v2) + 1e-8)
            feats.append(float(np.arccos(np.clip(cos_a, -1, 1))))
            valid_mask.append(True)
        else:
            feats.append(np.pi)
            valid_mask.append(False)

        # Right elbow angle
        if r_shoulder is not None and r_elbow is not None and r_wrist is not None:
            v1 = r_shoulder[:2] - r_elbow[:2]
            v2 = r_wrist[:2] - r_elbow[:2]
            cos_a = np.dot(v1, v2) / (np.linalg.norm(v1) * np.linalg.norm(v2) + 1e-8)
            feats.append(float(np.arccos(np.clip(cos_a, -1, 1))))
            valid_mask.append(True)
        else:
            feats.append(np.pi)
            valid_mask.append(False)

        # Left wrist to face distance (phone/makeup)
        if l_wrist is not None and nose is not None:
            feats.append(cls._dist(l_wrist[:2], nose[:2]) / shoulder_w)
            valid_mask.append(True)
        elif l_wrist is not None and p_nose is not None:
            feats.append(cls._dist(l_wrist[:2], p_nose[:2]) / shoulder_w)
            valid_mask.append(True)
        else:
            feats.append(2.0)
            valid_mask.append(False)

        # Right wrist to face distance
        if r_wrist is not None and nose is not None:
            feats.append(cls._dist(r_wrist[:2], nose[:2]) / shoulder_w)
            valid_mask.append(True)
        elif r_wrist is not None and p_nose is not None:
            feats.append(cls._dist(r_wrist[:2], p_nose[:2]) / shoulder_w)
            valid_mask.append(True)
        else:
            feats.append(2.0)
            valid_mask.append(False)

        # Left wrist x position (relative to shoulder center)
        if l_wrist is not None and l_shoulder is not None and r_shoulder is not None:
            mid_x = (l_shoulder[0] + r_shoulder[0]) / 2
            feats.append((l_wrist[0] - mid_x) / shoulder_w)
            valid_mask.append(True)
        else:
            feats.append(0.0)
            valid_mask.append(False)

        # Right wrist x position (relative to shoulder center)
        if r_wrist is not None and l_shoulder is not None and r_shoulder is not None:
            mid_x = (l_shoulder[0] + r_shoulder[0]) / 2
            feats.append((r_wrist[0] - mid_x) / shoulder_w)
            valid_mask.append(True)
        else:
            feats.append(0.0)
            valid_mask.append(False)

        # Wrist separation distance (hands together = phone use)
        if l_wrist is not None and r_wrist is not None:
            feats.append(cls._dist(l_wrist[:2], r_wrist[:2]) / shoulder_w)
            valid_mask.append(True)
        else:
            feats.append(1.0)
            valid_mask.append(False)

        # Head to shoulder midpoint vertical offset (looking down)
        if p_nose is not None and l_shoulder is not None and r_shoulder is not None:
            mid_y = (l_shoulder[1] + r_shoulder[1]) / 2
            feats.append((p_nose[1] - mid_y) / shoulder_w)
            valid_mask.append(True)
        else:
            feats.append(-1.0)
            valid_mask.append(False)

        # --- Hand detail features (8 features) ---
        # Left hand openness (spread of fingers)
        lh_wrist = g(left_hand_pts, cls.HAND_WRIST, left_valid, 2)
        lh_index = g(left_hand_pts, cls.HAND_INDEX_TIP, left_valid, 2)
        lh_middle = g(left_hand_pts, cls.HAND_MIDDLE_TIP, left_valid, 2)
        lh_pinky = g(left_hand_pts, cls.HAND_PINKY_TIP, left_valid, 2)

        if lh_wrist is not None and lh_index is not None:
            feats.append(cls._dist(lh_wrist, lh_index))
            valid_mask.append(True)
        else:
            feats.append(0.0)
            valid_mask.append(False)

        if lh_wrist is not None and lh_middle is not None:
            feats.append(cls._dist(lh_wrist, lh_middle))
            valid_mask.append(True)
        else:
            feats.append(0.0)
            valid_mask.append(False)

        # Finger spread left
        if lh_index is not None and lh_pinky is not None:
            feats.append(cls._dist(lh_index, lh_pinky))
            valid_mask.append(True)
        else:
            feats.append(0.0)
            valid_mask.append(False)

        # Left hand present flag
        feats.append(1.0 if lh_wrist is not None else 0.0)
        valid_mask.append(True)

        # Right hand
        rh_wrist = g(right_hand_pts, cls.HAND_WRIST, right_valid, 2)
        rh_index = g(right_hand_pts, cls.HAND_INDEX_TIP, right_valid, 2)
        rh_middle = g(right_hand_pts, cls.HAND_MIDDLE_TIP, right_valid, 2)
        rh_pinky = g(right_hand_pts, cls.HAND_PINKY_TIP, right_valid, 2)

        if rh_wrist is not None and rh_index is not None:
            feats.append(cls._dist(rh_wrist, rh_index))
            valid_mask.append(True)
        else:
            feats.append(0.0)
            valid_mask.append(False)

        if rh_wrist is not None and rh_middle is not None:
            feats.append(cls._dist(rh_wrist, rh_middle))
            valid_mask.append(True)
        else:
            feats.append(0.0)
            valid_mask.append(False)

        # Finger spread right
        if rh_index is not None and rh_pinky is not None:
            feats.append(cls._dist(rh_index, rh_pinky))
            valid_mask.append(True)
        else:
            feats.append(0.0)
            valid_mask.append(False)

        # Right hand present flag
        feats.append(1.0 if rh_wrist is not None else 0.0)
        valid_mask.append(True)

        return np.array(feats, dtype=np.float32), np.array(valid_mask, dtype=bool)


# ---------------------------------------------------------------------------
# Dataset
# ---------------------------------------------------------------------------

class HybridMultimodalDataset(Dataset):
    def __init__(
        self,
        records,
        img_root: Path,
        transform=None,
        landmark_path: Optional[Path] = None,
        pose_points: int = 33,
        face_points: int = 478,
        hand_points: int = 21,
        landmark_dims: int = 3,
        num_geo_features: int = 38,
    ):
        self.records = list(records)
        self.img_root = Path(img_root)
        self.transform = transform
        self.face_points = face_points
        self.hand_points = hand_points
        self.pose_points = pose_points
        self.landmark_dims = landmark_dims
        self.num_geo_features = num_geo_features
        self.landmark_lookup = load_landmark_lookup(landmark_path)
        self.geo_extractor = GeometricFeatureExtractor()

        if landmark_path is not None:
            print(f"Loaded landmark entries: {len(self.landmark_lookup)} from {landmark_path}")
        else:
            print("No landmark file found; using zero-padded landmark tensors.")

    def __len__(self):
        return len(self.records)

    def _load_image(self, idx):
        img_path = self.img_root / self.records[idx]["rel_path"]
        return Image.open(img_path).convert("RGB")

    def _lookup_landmarks(self, rec):
        key_candidates = [
            rec["rel_path"],
            Path(rec["rel_path"]).as_posix(),
            Path(rec["rel_path"]).name,
            Path(rec["rel_path"]).stem,
            rec["sample_id"],
        ]
        for key in key_candidates:
            key = normalize_lookup_key(key)
            if key in self.landmark_lookup:
                return self.landmark_lookup[key]
        return None

    def _coerce_points(self, raw_points, max_points: int):
        points = np.zeros((max_points, self.landmark_dims), dtype=np.float32)
        valid = np.zeros(max_points, dtype=bool)

        if raw_points is None:
            return points, valid

        if isinstance(raw_points, dict):
            if any(k in raw_points for k in ("left", "right")):
                raw_points = [raw_points.get("left"), raw_points.get("right")]
            elif "landmarks" in raw_points:
                raw_points = raw_points["landmarks"]
            else:
                raw_points = [raw_points]

        processed = []
        if isinstance(raw_points, (list, tuple)):
            if len(raw_points) > 0 and isinstance(raw_points[0], dict):
                for point in raw_points:
                    processed.append([
                        float(point.get("x", 0.0)),
                        float(point.get("y", 0.0)),
                        float(point.get("z", 0.0)),
                    ])
            elif len(raw_points) > 0 and isinstance(raw_points[0], (list, tuple)):
                for point in raw_points:
                    point = list(point)
                    while len(point) < self.landmark_dims:
                        point.append(0.0)
                    processed.append([float(v) for v in point[:self.landmark_dims]])
            else:
                flat = [float(v) for v in raw_points]
                usable = (len(flat) // self.landmark_dims) * self.landmark_dims
                flat = flat[:usable]
                if usable:
                    arr = np.asarray(flat, dtype=np.float32).reshape(-1, self.landmark_dims)
                    processed = arr.tolist()
        else:
            try:
                arr = np.asarray(raw_points, dtype=np.float32)
                if arr.ndim == 1:
                    usable = (arr.size // self.landmark_dims) * self.landmark_dims
                    arr = arr[:usable].reshape(-1, self.landmark_dims) if usable else np.zeros((0, self.landmark_dims))
                elif arr.ndim >= 2:
                    arr = arr.reshape(-1, arr.shape[-1])[:, :self.landmark_dims]
                processed = arr.tolist()
            except Exception:
                processed = []

        if processed:
            arr_np = np.array(processed[:max_points], dtype=np.float32)
            n = min(max_points, arr_np.shape[0])
            points[:n, :min(arr_np.shape[1], self.landmark_dims)] = arr_np[:n, :self.landmark_dims]
            valid[:n] = True

        return points, valid

    def _extract_all_landmarks(self, entry):
        if entry is None:
            return (
                np.zeros((self.pose_points, self.landmark_dims), dtype=np.float32),
                np.zeros(self.pose_points, dtype=bool),
                np.zeros((self.face_points, self.landmark_dims), dtype=np.float32),
                np.zeros(self.face_points, dtype=bool),
                np.zeros((self.hand_points, self.landmark_dims), dtype=np.float32),
                np.zeros(self.hand_points, dtype=bool),
                np.zeros((self.hand_points, self.landmark_dims), dtype=np.float32),
                np.zeros(self.hand_points, dtype=bool),
            )

        container = entry
        if isinstance(entry, dict) and "landmarks" in entry and isinstance(entry["landmarks"], dict):
            container = entry["landmarks"]

        pose_raw = None; face_raw = None; left_raw = None; right_raw = None

        if isinstance(container, dict):
            pose_raw = container.get("pose") or container.get("pose_landmarks") or container.get("pose_points")
            face_raw = container.get("face") or container.get("face_landmarks") or container.get("face_mesh") or container.get("face_points")
            left_raw = container.get("left_hand") or container.get("left_hand_landmarks") or container.get("hand_left")
            right_raw = container.get("right_hand") or container.get("right_hand_landmarks") or container.get("hand_right")

            hands_value = container.get("hands") or container.get("hand") or container.get("hand_landmarks")
            if isinstance(hands_value, dict):
                left_raw = left_raw or hands_value.get("left") or hands_value.get("left_hand")
                right_raw = right_raw or hands_value.get("right") or hands_value.get("right_hand")
            elif left_raw is None and right_raw is None and hands_value is not None:
                combined_pts, combined_v = self._coerce_points(hands_value, self.hand_points * 2)
                left_pts = combined_pts[:self.hand_points]
                right_pts = combined_pts[self.hand_points:]
                left_v = combined_v[:self.hand_points]
                right_v = combined_v[self.hand_points:]
                pose_pts, pose_v = self._coerce_points(pose_raw, self.pose_points)
                face_pts, face_v = self._coerce_points(face_raw, self.face_points)
                return pose_pts, pose_v, face_pts, face_v, left_pts, left_v, right_pts, right_v

        pose_pts, pose_v = self._coerce_points(pose_raw, self.pose_points)
        face_pts, face_v = self._coerce_points(face_raw, self.face_points)
        left_pts, left_v = self._coerce_points(left_raw, self.hand_points)
        right_pts, right_v = self._coerce_points(right_raw, self.hand_points)

        return pose_pts, pose_v, face_pts, face_v, left_pts, left_v, right_pts, right_v

    def __getitem__(self, idx):
        rec = self.records[idx]
        img = self._load_image(idx)
        y = int(rec["label"])

        entry = self._lookup_landmarks(rec)
        pose_pts, pose_v, face_pts, face_v, left_pts, left_v, right_pts, right_v = self._extract_all_landmarks(entry)

        # Compute geometric features
        geo_feats, geo_valid = self.geo_extractor.compute(
            face_pts, face_v, pose_pts, pose_v,
            left_pts, left_v, right_pts, right_v,
        )

        # Pad or truncate to expected size
        if len(geo_feats) < self.num_geo_features:
            geo_feats = np.pad(geo_feats, (0, self.num_geo_features - len(geo_feats)))
            geo_valid = np.pad(geo_valid, (0, self.num_geo_features - len(geo_valid)))
        else:
            geo_feats = geo_feats[:self.num_geo_features]
            geo_valid = geo_valid[:self.num_geo_features]

        x = self.transform(img) if self.transform else transforms.ToTensor()(img)

        return (
            x,
            torch.from_numpy(geo_feats),
            torch.from_numpy(geo_valid.astype(np.float32)),
        ), y


# Build datasets and loaders
train_ds = HybridMultimodalDataset(
    train_records, img_root=IMG_ROOT, transform=train_tfms,
    landmark_path=LANDMARK_PATH,
    face_points=cfg.face_points, hand_points=cfg.hand_points,
    pose_points=cfg.pose_points, landmark_dims=cfg.landmark_dims,
    num_geo_features=cfg.num_geo_features,
)
val_ds = HybridMultimodalDataset(
    val_records, img_root=IMG_ROOT, transform=val_tfms,
    landmark_path=LANDMARK_PATH,
    face_points=cfg.face_points, hand_points=cfg.hand_points,
    pose_points=cfg.pose_points, landmark_dims=cfg.landmark_dims,
    num_geo_features=cfg.num_geo_features,
)

pin_memory = (device.type == "cuda")
train_loader = DataLoader(
    train_ds, batch_size=cfg.batch_size, shuffle=True,
    num_workers=cfg.num_workers, pin_memory=pin_memory,
    persistent_workers=(cfg.num_workers > 0),
)
val_loader = DataLoader(
    val_ds, batch_size=cfg.batch_size, shuffle=False,
    num_workers=cfg.num_workers, pin_memory=pin_memory,
    persistent_workers=(cfg.num_workers > 0),
)

print("Train dataset size:", len(train_ds))
print("Val dataset size:  ", len(val_ds))

t0 = time.time()
batch = next(iter(train_loader))
(sample_images, sample_geo, sample_geo_valid), sample_labels = batch
print(
    "Got batch:",
    "images", sample_images.shape,
    "geo_feats", sample_geo.shape,
    "labels", sample_labels.shape,
    "in", round(time.time() - t0, 2), "sec",
)

## 7) Model

In [ ]:
class HybridCNNLandmarkTransformer(nn.Module):
    """
    Hybrid architecture: single lightweight CNN backbone for image features +
    geometric landmark features, fused via a small transformer encoder.

    Key design choices:
    - MobileNetV3-Small (not Large) for mobile-friendly inference
    - Geometric features (EAR, MAR, head pose, hand positions) instead of raw landmarks
    - Lightweight transformer learns body-part relationships for classification
    - Single CLS token output for final prediction
    """

    def __init__(
        self,
        num_classes: int,
        num_geo_features: int = 38,
        embed_dim: int = 128,
        num_heads: int = 4,
        num_layers: int = 4,
        ff_dim: int = 384,
        dropout: float = 0.15,
    ):
        super().__init__()
        self.embed_dim = embed_dim
        self.num_geo_features = num_geo_features

        # --- Image encoder: MobileNetV3-Small ---
        self.image_encoder = mobilenet_v3_small(weights=MobileNet_V3_Small_Weights.DEFAULT)
        image_out_dim = self.image_encoder.classifier[0].in_features  # 576
        self.image_encoder.classifier = nn.Identity()

        # Freeze early layers, fine-tune last 3 blocks
        for param in self.image_encoder.features.parameters():
            param.requires_grad = False
        for block in self.image_encoder.features[-3:]:
            for param in block.parameters():
                param.requires_grad = True

        # Pool CNN feature map to 2x2 = 4 spatial tokens
        self.num_image_tokens = 4
        self.image_pool = nn.AdaptiveAvgPool2d((2, 2))
        self.image_proj = nn.Sequential(
            nn.Linear(image_out_dim, embed_dim),
            nn.LayerNorm(embed_dim),
            nn.GELU(),
        )
        self.image_pos = nn.Parameter(torch.zeros(1, self.num_image_tokens, embed_dim))

        # --- Geometric feature encoder ---
        # Project each geo feature to a token, group them into semantic chunks
        self.num_geo_groups = 5  # head_pose, eyes, mouth, pose/body, hands
        self.geo_group_sizes = [6, 2, 1, 12, 8]  # must sum to >= num_geo_features

        # One MLP per group to produce one token each
        self.geo_encoders = nn.ModuleList()
        offset = 0
        for gs in self.geo_group_sizes:
            actual_size = min(gs, num_geo_features - offset) if offset < num_geo_features else 1
            self.geo_encoders.append(nn.Sequential(
                nn.Linear(actual_size, embed_dim),
                nn.LayerNorm(embed_dim),
                nn.GELU(),
                nn.Dropout(dropout),
            ))
            offset += gs
        self.geo_group_pos = nn.Parameter(torch.zeros(1, self.num_geo_groups, embed_dim))

        # --- Transformer fusion ---
        self.cls_token = nn.Parameter(torch.zeros(1, 1, embed_dim))

        # Type embeddings to distinguish image vs landmark tokens
        self.image_type_embed = nn.Parameter(torch.zeros(1, 1, embed_dim))
        self.geo_type_embed = nn.Parameter(torch.zeros(1, 1, embed_dim))
        self.cls_type_embed = nn.Parameter(torch.zeros(1, 1, embed_dim))

        encoder_layer = nn.TransformerEncoderLayer(
            d_model=embed_dim,
            nhead=num_heads,
            dim_feedforward=ff_dim,
            dropout=dropout,
            activation="gelu",
            batch_first=True,
            norm_first=True,
        )
        self.transformer = nn.TransformerEncoder(encoder_layer, num_layers=num_layers)

        # --- Classification head ---
        self.head = nn.Sequential(
            nn.LayerNorm(embed_dim),
            nn.Linear(embed_dim, 256),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(256, num_classes),
        )

        self._init_parameters()

    def _init_parameters(self):
        for p in [self.cls_token, self.image_pos, self.geo_group_pos,
                   self.image_type_embed, self.geo_type_embed, self.cls_type_embed]:
            nn.init.normal_(p, std=0.02)

    def forward(self, images, geo_feats, geo_valid):
        B = images.size(0)

        # --- Image tokens ---
        feat_map = self.image_encoder.features(images)   # [B, 576, H, W]
        feat_map = self.image_pool(feat_map)              # [B, 576, 2, 2]
        img_tokens = feat_map.flatten(2).transpose(1, 2)  # [B, 4, 576]
        img_tokens = self.image_proj(img_tokens)           # [B, 4, embed_dim]
        img_tokens = img_tokens + self.image_pos + self.image_type_embed

        # --- Geometric feature tokens (one per semantic group) ---
        geo_tokens = []
        offset = 0
        for i, gs in enumerate(self.geo_group_sizes):
            end = min(offset + gs, self.num_geo_features)
            if offset >= self.num_geo_features:
                chunk = torch.zeros(B, 1, dtype=geo_feats.dtype, device=geo_feats.device)
            else:
                chunk = geo_feats[:, offset:end]
                actual = end - offset
                if actual < gs:
                    pad = torch.zeros(B, gs - actual, dtype=geo_feats.dtype, device=geo_feats.device)
                    chunk = torch.cat([chunk, pad], dim=1)
            token = self.geo_encoders[i](chunk)  # [B, embed_dim]
            geo_tokens.append(token.unsqueeze(1))
            offset += gs

        geo_tokens = torch.cat(geo_tokens, dim=1)  # [B, num_geo_groups, embed_dim]
        geo_tokens = geo_tokens + self.geo_group_pos + self.geo_type_embed

        # Scale geo tokens by average validity (soft gating)
        geo_validity_score = geo_valid.mean(dim=1, keepdim=True).unsqueeze(-1)  # [B, 1, 1]
        geo_tokens = geo_tokens * (0.5 + 0.5 * geo_validity_score)

        # --- CLS token ---
        cls_token = self.cls_token.expand(B, -1, -1) + self.cls_type_embed

        # --- Assemble and run transformer ---
        # [CLS, img_1..img_4, geo_1..geo_5] = 10 tokens total
        tokens = torch.cat([cls_token, img_tokens, geo_tokens], dim=1)

        encoded = self.transformer(tokens)
        logits = self.head(encoded[:, 0, :])  # CLS token output
        return logits


def build_model(num_classes: int, device: torch.device):
    model = HybridCNNLandmarkTransformer(
        num_classes=num_classes,
        num_geo_features=cfg.num_geo_features,
        embed_dim=cfg.embed_dim,
        num_heads=cfg.num_heads,
        num_layers=cfg.num_layers,
        ff_dim=cfg.ff_dim,
        dropout=cfg.dropout,
    )
    model = model.to(device)

    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    total = sum(p.numel() for p in model.parameters())
    print(f"Trainable params: {trainable:,} / {total:,}")

    return model

model = build_model(NUM_CLASSES, device)

## 8) Training utilities

In [ ]:
criterion = nn.CrossEntropyLoss(label_smoothing=0.05)

backbone_params = []
head_params = []

for name, p in model.named_parameters():
    if not p.requires_grad:
        continue
    if "image_encoder.features" in name:
        backbone_params.append(p)
    else:
        head_params.append(p)

print(f"Backbone params: {sum(p.numel() for p in backbone_params):,}")
print(f"Head params: {sum(p.numel() for p in head_params):,}")

optimizer = torch.optim.AdamW([
    {"params": backbone_params, "lr": cfg.backbone_lr},
    {"params": head_params, "lr": cfg.head_lr},
], weight_decay=cfg.weight_decay)

# Cosine annealing (better for transformers than step LR)
train_scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
    optimizer,
    T_max=cfg.epochs - cfg.warm_epochs,
    eta_min=1e-6,
)

class WarmUpLR(torch.optim.lr_scheduler._LRScheduler):
    def __init__(self, optimizer, total_iters, last_epoch=-1):
        self.total_iters = total_iters
        super().__init__(optimizer, last_epoch)

    def get_lr(self):
        return [base_lr * self.last_epoch / (self.total_iters + 1e-8) for base_lr in self.base_lrs]

warmup_scheduler = WarmUpLR(optimizer, cfg.warm_epochs * len(train_loader))


def train_one_epoch(epoch):
    model.train()
    loss_train = 0.0
    correct_prediction = 0

    start = time.time()

    for batch_index, batch in enumerate(tqdm(train_loader, desc=f"train {epoch}", leave=False)):
        (images, geo_feats, geo_valid), labels = batch

        images = images.to(device, non_blocking=True)
        geo_feats = geo_feats.to(device, non_blocking=True)
        geo_valid = geo_valid.to(device, non_blocking=True)
        labels = labels.to(device, non_blocking=True)

        optimizer.zero_grad()
        outputs = model(images, geo_feats, geo_valid)
        loss = criterion(outputs, labels)
        loss.backward()

        # Gradient clipping for transformer stability
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)

        optimizer.step()

        if epoch <= cfg.warm_epochs:
            warmup_scheduler.step()

        loss_train += loss.item()
        _, predicted = torch.max(outputs.data, 1)
        correct_prediction += (predicted == labels).sum().item()

    train_loss = loss_train / len(train_loader)
    train_acc = correct_prediction / len(train_ds)

    finish = time.time()
    print(f"epoch {epoch} training time consumed: {finish - start:.2f}s")

    return train_loss, train_acc


def validate_one_epoch(epoch):
    model.eval()
    test_loss = 0.0
    correct = 0

    y_true = []
    y_pred = []

    start = time.time()

    with torch.no_grad():
        for batch in tqdm(val_loader, desc=f"val {epoch}", leave=False):
            (images, geo_feats, geo_valid), labels = batch

            images = images.to(device, non_blocking=True)
            geo_feats = geo_feats.to(device, non_blocking=True)
            geo_valid = geo_valid.to(device, non_blocking=True)
            labels = labels.to(device, non_blocking=True)

            outputs = model(images, geo_feats, geo_valid)
            loss = criterion(outputs, labels)

            test_loss += loss.item()
            preds = outputs.argmax(1)

            correct += preds.eq(labels).sum().item()
            y_true.append(labels.cpu().numpy())
            y_pred.append(preds.cpu().numpy())

    finish = time.time()
    test_loss = test_loss / len(val_loader)
    test_acc = correct / len(val_ds)

    y_true = np.concatenate(y_true)
    y_pred = np.concatenate(y_pred)
    test_f1 = f1_score(y_true, y_pred, average="macro")

    print(
        f"Evaluating Network.....\n"
        f"Val set: Average loss: {test_loss:.4f}, "
        f"Accuracy: {test_acc:.4f}, "
        f"Macro F1: {test_f1:.4f}, "
        f"Time consumed:{finish - start:.2f}s"
    )

    return test_loss, test_acc, test_f1

## 9) Train

In [ ]:
best_f1 = float("-inf")
history = []

best_path = cfg.out_dir / cfg.best_name

for epoch in range(1, cfg.epochs + 1):

    train_loss, train_acc = train_one_epoch(epoch)
    val_loss, val_acc, val_f1 = validate_one_epoch(epoch)

    if epoch > cfg.warm_epochs:
        train_scheduler.step()

    history.append({
        "epoch": epoch,
        "train_loss": train_loss,
        "train_acc": train_acc,
        "val_loss": val_loss,
        "val_acc": val_acc,
        "val_f1": val_f1,
        "backbone_lr": optimizer.param_groups[0]["lr"],
        "head_lr": optimizer.param_groups[1]["lr"],
    })

    print(
        f"Epoch {epoch:02d}/{cfg.epochs} | "
        f"train loss {train_loss:.4f} acc {train_acc:.4f} | "
        f"val loss {val_loss:.4f} acc {val_acc:.4f} f1 {val_f1:.4f}"
    )

    if val_f1 > best_f1:
        best_f1 = val_f1
        torch.save(model.state_dict(), best_path)
        print("best model by macro F1! save...", best_path)

    if epoch % cfg.save_every == 0:
        ckpt_path = cfg.out_dir / f"hybrid_transformer_epoch_{epoch}.pt"
        torch.save(model.state_dict(), ckpt_path)

## 10) Training results

In [ ]:
hist = pd.DataFrame(history)
hist.to_csv(cfg.out_dir / "hybrid_transformer_history.csv", index=False)
hist.tail()

In [ ]:
if not hist.empty:
    fig, axes = plt.subplots(1, 3, figsize=(16, 4))

    axes[0].plot(hist["epoch"], hist["train_loss"], label="train_loss")
    axes[0].plot(hist["epoch"], hist["val_loss"], label="val_loss")
    axes[0].legend()
    axes[0].set_xlabel("epoch")
    axes[0].set_ylabel("loss")
    axes[0].set_title("Loss")

    axes[1].plot(hist["epoch"], hist["train_acc"], label="train_acc")
    axes[1].plot(hist["epoch"], hist["val_acc"], label="val_acc")
    axes[1].legend()
    axes[1].set_xlabel("epoch")
    axes[1].set_ylabel("accuracy")
    axes[1].set_title("Accuracy")

    axes[2].plot(hist["epoch"], hist["backbone_lr"], label="backbone_lr")
    axes[2].plot(hist["epoch"], hist["head_lr"], label="head_lr")
    axes[2].legend()
    axes[2].set_xlabel("epoch")
    axes[2].set_ylabel("learning rate")
    axes[2].set_title("Learning Rate (Cosine Annealing)")

    plt.tight_layout()
    plt.show()

## 11) Best-model evaluation

In [ ]:
best_model = build_model(NUM_CLASSES, device)

state_dict = torch.load(best_path, map_location=device)
best_model.load_state_dict(state_dict)
best_model = best_model.to(device).eval()

y_true_all, y_pred_all = [], []
with torch.no_grad():
    for batch in tqdm(val_loader, desc="val eval"):
        (images, geo_feats, geo_valid), labels = batch

        images = images.to(device, non_blocking=True)
        geo_feats = geo_feats.to(device, non_blocking=True)
        geo_valid = geo_valid.to(device, non_blocking=True)
        labels = labels.to(device, non_blocking=True)

        logits = best_model(images, geo_feats, geo_valid)

        preds = logits.argmax(1).cpu().numpy()
        y_pred_all.append(preds)
        y_true_all.append(labels.cpu().numpy())

y_true_all = np.concatenate(y_true_all)
y_pred_all = np.concatenate(y_pred_all)

print("Accuracy:", accuracy_score(y_true_all, y_pred_all))
print("Macro F1:", f1_score(y_true_all, y_pred_all, average="macro"))
print("\nClassification report:\n")
print(classification_report(y_true_all, y_pred_all, target_names=CLASS_NAMES, digits=4))

cm = confusion_matrix(y_true_all, y_pred_all)
plt.figure(figsize=(10, 10))
plt.imshow(cm, interpolation="nearest", cmap="Blues")
plt.title("Confusion Matrix")
plt.colorbar()
tick_marks = np.arange(len(CLASS_NAMES))
plt.xticks(tick_marks, CLASS_NAMES, rotation=90)
plt.yticks(tick_marks, CLASS_NAMES)
plt.ylabel("True label")
plt.xlabel("Predicted label")
plt.tight_layout()
plt.show()

## 12) Quick inference preview

In [ ]:
def denorm(x, mean, std):
    mean_t = torch.tensor(mean).view(3, 1, 1)
    std_t = torch.tensor(std).view(3, 1, 1)
    return (x * std_t) + mean_t

best_model.eval()
sample_count = min(8, len(val_ds))
sample_idx = np.random.choice(len(val_ds), size=sample_count, replace=False)
samples = [val_ds[i] for i in sample_idx]

xb = torch.stack([s[0][0] for s in samples]).to(device)
geo_b = torch.stack([s[0][1] for s in samples]).to(device)
geo_v_b = torch.stack([s[0][2] for s in samples]).to(device)

yb = np.array([s[1] for s in samples])

with torch.no_grad():
    logits = best_model(xb, geo_b, geo_v_b)
    probs = torch.softmax(logits, dim=1)
    pred = probs.argmax(1).cpu().numpy()

plt.figure(figsize=(12, 6))
for i in range(sample_count):
    ax = plt.subplot(2, math.ceil(sample_count / 2), i + 1)
    img = denorm(samples[i][0][0].cpu(), mean, std).clamp(0, 1)
    ax.imshow(img.permute(1, 2, 0))
    ax.axis("off")
    ax.set_title(f"T:{CLASS_NAMES[yb[i]]}\nP:{CLASS_NAMES[pred[i]]}", fontsize=9)
plt.tight_layout()
plt.show()